In [1]:
import os
import re
import time
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from scipy import stats

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import ContextualCompressionRetriever, EnsembleRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from dotenv import load_dotenv

load_dotenv()

MODEL = "gpt-4o-mini"
llm = ChatOpenAI(model=MODEL)
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")


In [2]:
documents = [
    Document(page_content="트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.", metadata={"id": "d1"}),
    Document(page_content="BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.", metadata={"id": "d2"}),
    Document(page_content="GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.", metadata={"id": "d3"}),
    Document(page_content="RAG는 검색 증강 생성 기법으로 외부 지식을 LLM에 결합하여 할루시네이션을 줄입니다.", metadata={"id": "d4"}),
    Document(page_content="벡터 데이터베이스는 임베딩 벡터를 저장하고 유사도 기반 검색을 수행합니다. FAISS, Pinecone 등이 있습니다.", metadata={"id": "d5"}),
    Document(page_content="파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.", metadata={"id": "d6"}),
    Document(page_content="프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.", metadata={"id": "d7"}),
    Document(page_content="토큰화는 텍스트를 모델이 처리할 수 있는 단위로 분할하는 과정입니다. BPE, WordPiece 등이 사용됩니다.", metadata={"id": "d8"}),
]


In [3]:
vectorstore = FAISS.from_documents(documents, embeddings_model)
bm25_retriever = BM25Retriever.from_documents(documents, k=5)

In [4]:
doc_embeddings = {}

def get_embedding(text):
    return np.array(embeddings_model.embed_query(text))

for doc in doc_embeddings:
    doc_embeddings[doc.metadata['id']] = get_embedding(doc.page_content)

In [5]:
# query - doc1, doc2, ...   Cross-encoder
# 사용자 쿼리 : 트랜스포머와 BERT의 관계?
#     1) 트랜스포머는 ~~~~
    
#     2) BERT는 ~~~ 입니다

In [6]:
# query -> emb_query
# doc1 -> emb_doc1
# ..
# => 문장들간의 의미적 관련성

In [7]:
# 검색 -> doc1, doc2, ... doc5 -> 후보문서 -> re-rank ----(Cross-encoder / llm)--> re-ranked -> generate

In [8]:
# Cross-encoder
# 1. normal vector retriever
# query ---(emb_model)---> query_emb
# doc1 ----(emb_model) ---> doc1_emb

# # 2. cross-encoder retriever
# (query - doc1 ) ---(emb_model)---> (query_doc1_emb)
# (query - doc2 ) ---(emb_model)---> (query_doc2_emb)

# LLM
# query, doc1 --(LLM)---> score, t/f

In [9]:
def vector_search(query, vectorstore, top_k=5):
    results = vectorstore.similarity_search_with_score(query, k=top_k)
    return [(doc, 1.0 / (1.0 + score)) for doc, score in results]

query = "트랜스포머와 BERT의 관계"
results = vector_search(query, vectorstore)
results

[(Document(id='0820d82b-7f85-4f20-9278-df128485b595', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  np.float32(0.52375495)),
 (Document(id='f941a5a5-8291-41c4-97c4-8719bedacde4', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  np.float32(0.46395132)),
 (Document(id='9711ac83-c999-43eb-8556-3dd19c3ec446', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  np.float32(0.41890207)),
 (Document(id='08c38071-9fc0-463a-a01b-5651b4231e2b', metadata={'id': 'd6'}, page_content='파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.'),
  np.float32(0.41281974)),
 (Document(id='a81e6fa7-5295-4594-a085-23867b311ae3', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  np.float32(0.39723063))]

In [10]:
def keyword_rerank(query, search_results):
    query_terms = set(query.lower().split())
    reranked = []
    
    for doc, orig_score in search_results:
        doc_terms = doc.page_content.lower().split()
        keyword_hits = sum(1 for t in doc_terms if t in query_terms)   # [1, 1]
        new_score = orig_score + 0.1 * keyword_hits
        reranked.append((doc, new_score, orig_score))
        
    reranked.sort(key = lambda x: x[1], reverse=True)
    return reranked

In [11]:
reranked = keyword_rerank(query, results)
reranked

[(Document(id='0820d82b-7f85-4f20-9278-df128485b595', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  np.float32(0.52375495),
  np.float32(0.52375495)),
 (Document(id='f941a5a5-8291-41c4-97c4-8719bedacde4', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  np.float32(0.46395132),
  np.float32(0.46395132)),
 (Document(id='9711ac83-c999-43eb-8556-3dd19c3ec446', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  np.float32(0.41890207),
  np.float32(0.41890207)),
 (Document(id='08c38071-9fc0-463a-a01b-5651b4231e2b', metadata={'id': 'd6'}, page_content='파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.'),
  np.float32(0.41281974),
  np.float32(0.41281974)),
 (Document(id='a81e6fa7-5295-4594-a085-23867b311ae3', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  np.float32(0.39723063),
  np.float32(0.397

In [12]:
def rank_change(original_results, reranked_results):
    
    orig_ranks = {doc_.metadata['id']: i+1 for i, (doc_, _) in enumerate(original_results)}
    new_ranks = {doc_.metadata['id']: i+1 for i, (doc_, _, _) in enumerate(reranked_results)}
    
    changes = []
    for doc_id in orig_ranks:
        old_r = orig_ranks[doc_id]
        new_r = new_ranks.get(doc_id, -1)
        
        changes.append({
            'doc_id' : doc_id,
            'before' : old_r,
            'after' : new_r,
            'change' : old_r - new_r
        })
        
    return pd.DataFrame(changes).sort_values('after')


In [13]:
df = rank_change(results, reranked)
df

,doc_id,before,after,change
0,d1,1,1,0
1,d2,2,2,0
2,d3,3,3,0
3,d6,4,4,0
4,d7,5,5,0


In [14]:
# ranked -> (Cross-encoder / LLM) -> reranked

In [15]:
# vector search : 쿼리 / 문서 : Bi-encoder

In [16]:
# BM25 -> rerank 
# (쿼리 - 문서) -> rerank
# LLM -> rerank

In [17]:
# TF-IDF 
class BM25Reranker: 
    def __init__(self, k1 = 1.5, b=0.75):
        self.k1 = k1
        self.b = b
    
    def rerank(self, query, search_results):
        docs = [doc for doc, _ in search_results]
        tokenized = [doc.page_content.lower().split() for doc in docs]
        
        avg_dl = np.mean([len(t) for t in tokenized])
        
        df_count = Counter()
        for tokens in tokenized:
            for t in set(tokens):
                df_count[t] += 1
        N = len(docs)
        
        query_tokens = query.lower().split()
        scored = []
        
        for i, doc in enumerate(docs):
            score = 0.0
            doc_len = len(tokenized[i])
            tf_count = Counter(tokenized[i])
            
            for qt in query_tokens :
                tf = tf_count.get(qt, 0)
                if tf == 0:
                    continue
                
                df = df_count.get(qt, 0)
                idf = math.log((N-df + 0.5) / (df + 0.5) + 1)    # tf-idf : tf x idf   10 -> log(10) : 1 (0.1 -> (-1))
                numerator = tf * (self.k1 + 1)
                denominator = tf + self.k1 * (1 - self.b + self.b * doc_len /avg_dl)
                score +=  idf * numerator / denominator
        
            scored.append((doc, score))
        
        scored.sort(key = lambda x: x[1], reverse=True)
        return scored
                
        
        

In [18]:
bm25_reranker = BM25Reranker()
bm25_results = bm25_reranker.rerank(query, results)

In [19]:
bm25_results

[(Document(id='0820d82b-7f85-4f20-9278-df128485b595', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  0.0),
 (Document(id='f941a5a5-8291-41c4-97c4-8719bedacde4', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  0.0),
 (Document(id='9711ac83-c999-43eb-8556-3dd19c3ec446', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  0.0),
 (Document(id='08c38071-9fc0-463a-a01b-5651b4231e2b', metadata={'id': 'd6'}, page_content='파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.'),
  0.0),
 (Document(id='a81e6fa7-5295-4594-a085-23867b311ae3', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  0.0)]

In [20]:
# ['d1'] 프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.
# ['d2'] 프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.

In [21]:
def llm_rerank(query, search_results, top_k=3):
    docs_text = '\n'.join(
        f"[{doc.metadata['id']}] {doc.page_content}" for doc, _ in search_results
    )
    
    score_template = ", ".join(f'"{doc.metadata["id"]}": 0.0-1.0' for doc, _ in search_results)
    
    scoring_chain = ChatPromptTemplate.from_messages([
        ('system', '당신은 scoring 시스템 입니다. 항상 json 형태로 출력하세요'),
        ('human', """다음 쿼리에 대해 각 문서의 관련성을 0.0~1.0으로 평가하세요.
        
        쿼리 : {query}
        
        문서들 :
        {docs_text}
        
        "스트링으로 묶지말고" JSON으로 답하세요:
        {{"scores" : {{{score_template}}}}}""")
    ]) | llm | StrOutputParser()
    
    result = scoring_chain.invoke({
        'query' : query,
        'docs_text' : docs_text,
        'score_template' : score_template
    })
    
    cleaned = result.strip()
    if cleaned.startswith('```json'):
        cleaned = cleaned.replace('```json', '')
        cleaned = cleaned.replace('```', '')
        
    parsed = json.loads(cleaned)
    scores = parsed.get('scores', {})
    
    scored = []
    
    for doc, orig in search_results:
        rerank_score = scores.get(doc.metadata['id'],0.0)
        scored.append((doc, float(rerank_score), orig))
        
    scored.sort(key = lambda x : x[1], reverse=True)
    return scored[:top_k]

In [22]:
query # BERT : 

'트랜스포머와 BERT의 관계'

In [23]:
results

[(Document(id='0820d82b-7f85-4f20-9278-df128485b595', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  np.float32(0.52375495)),
 (Document(id='f941a5a5-8291-41c4-97c4-8719bedacde4', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  np.float32(0.46395132)),
 (Document(id='9711ac83-c999-43eb-8556-3dd19c3ec446', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  np.float32(0.41890207)),
 (Document(id='08c38071-9fc0-463a-a01b-5651b4231e2b', metadata={'id': 'd6'}, page_content='파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.'),
  np.float32(0.41281974)),
 (Document(id='a81e6fa7-5295-4594-a085-23867b311ae3', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  np.float32(0.39723063))]

In [24]:
llm_results = llm_rerank(query, results, top_k=3)

In [25]:
llm_results

[(Document(id='f941a5a5-8291-41c4-97c4-8719bedacde4', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  1.0,
  np.float32(0.46395132)),
 (Document(id='9711ac83-c999-43eb-8556-3dd19c3ec446', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  0.5,
  np.float32(0.41890207)),
 (Document(id='0820d82b-7f85-4f20-9278-df128485b595', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  0.3,
  np.float32(0.52375495))]

In [26]:
def hybrid_rerank(query, search_results, bm25_weight = 0.3):
    bm25 = BM25Reranker()
    bm25_scored = bm25.rerank(query, search_results)
    bm25_map = {doc.metadata['id'] : score for doc, score in bm25_scored}
    
    llm_scored = llm_rerank(query, search_results, top_k = len(search_results))
    llm_map = {doc.metadata['id'] : score for doc, score, _ in llm_scored}
    
    def normalize(scores_dict):
        vals = list(scores_dict.values())
        min_, max_ = min(vals), max(vals)
        range_ = max_ - min_ if max_ > min_ else 1e-8
        return {k : (v - min_) / range_ for k, v in scores_dict.items()}
    
    bm25_norm = normalize(bm25_map)
    llm_norm = normalize(llm_map)
    
    combined = []
    for doc, _ in search_results:
        did = doc.metadata['id']
        score = bm25_weight * bm25_norm.get(did, 0) + (1- bm25_weight) * llm_norm.get(did, 0)
        
        combined.append((doc, score))
    combined.sort(key = lambda x:x[1], reverse=True)
    return combined

In [27]:
hybrid = hybrid_rerank(query, results)

In [28]:
hybrid

[(Document(id='f941a5a5-8291-41c4-97c4-8719bedacde4', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  0.7),
 (Document(id='9711ac83-c999-43eb-8556-3dd19c3ec446', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  0.27999999999999997),
 (Document(id='0820d82b-7f85-4f20-9278-df128485b595', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  0.21),
 (Document(id='08c38071-9fc0-463a-a01b-5651b4231e2b', metadata={'id': 'd6'}, page_content='파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.'),
  0.06999999999999999),
 (Document(id='a81e6fa7-5295-4594-a085-23867b311ae3', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  0.0)]

In [29]:
# pointwise reranking (1개씩 넣는다)
# - 정확, 이유를 명확하게 설명, 빠르다(병렬)
# - 비용, 문서간 비교를 못한다
# listwise reranking  (N개씩 넣는다 : 이 문서들을 정렬해줘, 스코어링해줘) 
# - 비용, 문서간 비교를 한다
# - 할루시네이션/불명확한 설명, 컨텍스트(많은 문서)

In [30]:
# 5개 -> listwise
# 빠른 응답이 필요한 경우 -> 1개씩

In [31]:
# RAG - Retriever (Reranker) -> generator
# threshold scoring : 0.5 이하는 올리지마  : fixed threshold

# dynamic threshold : mean - 1 * sigma (표준편차) : quantitle
# scoring gap : 점수차
# top_k : 

In [32]:
def llm_listwise_rerank(query, search_results):
    docs_text = '\n'.join(
            f"{i+1}. [{doc.metadata['id']}] {doc.page_content[:80]}" for i, (doc, _) in enumerate(search_results)
    )
    
    ranking_chain = ChatPromptTemplate.from_messages([
        ('system', "당신은 문서 랭킹 시스템입니다"),
        ('human', """다음 문서들을 쿼리와의 관련성 순서로 정렬하세요.
        
        쿼리 : {query}
        
        문서들 : {docs_text}
        
        가장 관련성 높은 순서대로 문서 번호를 쉼표로 나열하세요 (예: 3,1,5,2,4):""")
    ]) | llm | StrOutputParser()
    
    result = ranking_chain.invoke({'query' : query, 'docs_text' : docs_text})
    
    order = [int(x.strip()) for x in result.strip().split(',')]
    
    docs_list = [doc for doc, _ in search_results]
    reranked = []
    for rank, idx in enumerate(order):
        if 1<=idx<=len(docs_list):
            docs = docs_list[idx-1]
            reranked.append((docs, 1.0-rank*0.1))
            
    return reranked

In [33]:
listwise = llm_listwise_rerank(query, results)

In [34]:
listwise

[(Document(id='f941a5a5-8291-41c4-97c4-8719bedacde4', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  1.0),
 (Document(id='0820d82b-7f85-4f20-9278-df128485b595', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  0.9),
 (Document(id='9711ac83-c999-43eb-8556-3dd19c3ec446', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  0.8),
 (Document(id='08c38071-9fc0-463a-a01b-5651b4231e2b', metadata={'id': 'd6'}, page_content='파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.'),
  0.7),
 (Document(id='a81e6fa7-5295-4594-a085-23867b311ae3', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  0.6)]

In [45]:
class ScoreFilter:
    
    @staticmethod
    def fixed_threshold(scored_docs, threshold= 0.5):
        return [(doc, s) for doc, s in scored_docs if s >= threshold]
    
    @staticmethod
    def dynamic_threshold(scored_docs, std_factor) : 
        scores = [s for _, s in scored_docs]
        threshold = np.mean(scores) - std_factor * np.std(scores)
        return [(doc, s) for doc, s in scored_docs if s >= threshold]
    
    @staticmethod
    def score_gap(scored_docs, min_docs=2):
        if len(scored_docs) <= min_docs:
            return scored_docs
        
        scores = [s for _, s in scored_docs]
        gaps = [(scores[i] - scores[i+1], i+1) for i in range(len(scores)-1)]
        max_gap_idx = max(gaps, key=lambda x:x[0])[1]
        cut = max(min_docs, max_gap_idx)
        return scored_docs[:cut]



In [46]:
# sf = ScoreFilter()
# sf.abc()

In [47]:
test_scores = [(Document(page_content="", metadata = {'id': f'd{i}'}), s) for i, s in enumerate([0.9, 0.85, 0.7, 0.5, 0.2])]
test_scores

[(Document(metadata={'id': 'd0'}, page_content=''), 0.9),
 (Document(metadata={'id': 'd1'}, page_content=''), 0.85),
 (Document(metadata={'id': 'd2'}, page_content=''), 0.7),
 (Document(metadata={'id': 'd3'}, page_content=''), 0.5),
 (Document(metadata={'id': 'd4'}, page_content=''), 0.2)]

In [48]:
filtered = ScoreFilter.fixed_threshold(test_scores, threshold= 0.5)

In [49]:
filtered

[(Document(metadata={'id': 'd0'}, page_content=''), 0.9),
 (Document(metadata={'id': 'd1'}, page_content=''), 0.85),
 (Document(metadata={'id': 'd2'}, page_content=''), 0.7),
 (Document(metadata={'id': 'd3'}, page_content=''), 0.5)]

In [50]:
filtered = ScoreFilter.dynamic_threshold(test_scores, std_factor=1)
filtered

[(Document(metadata={'id': 'd0'}, page_content=''), 0.9),
 (Document(metadata={'id': 'd1'}, page_content=''), 0.85),
 (Document(metadata={'id': 'd2'}, page_content=''), 0.7),
 (Document(metadata={'id': 'd3'}, page_content=''), 0.5)]

In [51]:
filtered = ScoreFilter.score_gap(test_scores, min_docs=2)
filtered

[(Document(metadata={'id': 'd0'}, page_content=''), 0.9),
 (Document(metadata={'id': 'd1'}, page_content=''), 0.85),
 (Document(metadata={'id': 'd2'}, page_content=''), 0.7),
 (Document(metadata={'id': 'd3'}, page_content=''), 0.5)]

In [52]:
class AdaptiveFilter:
    
    # 점수의 분포에 따라서 score filtering 전략을 자동으로 선택
    
    @staticmethod # decorator
    def adapt_filter(scored_docs):
        scores = [s for _, s in scored_docs]
        std = np.std(scores)
        score_range = max(scores) - min(scores)
        
        if std > 0.2:
            result = ScoreFilter.score_gap(test_scores, min_docs=2)
        elif score_range < 0.1:
            result = scored_docs[:len(scored_docs)//2]
        else:
            result = ScoreFilter.dynamic_threshold(test_scores, std_factor=1)
        
        return result
    

In [53]:
filtered = AdaptiveFilter.adapt_filter(test_scores)
filtered

[(Document(metadata={'id': 'd0'}, page_content=''), 0.9),
 (Document(metadata={'id': 'd1'}, page_content=''), 0.85),
 (Document(metadata={'id': 'd2'}, page_content=''), 0.7),
 (Document(metadata={'id': 'd3'}, page_content=''), 0.5)]

In [ ]:
# Context compression
# gpt-4o mini : 128K 토큰 (system prompt, (context1, context2 ...), user_prompt, 답변을 위한 공간)
# 하네스 엔지니어링 : compress 전략  100k -> 압축해라, 턴이 20턴정도 끝나면 압축해라 ...

In [ ]:
# 에이전트 -> 잘하고 내가 편하다 -> 위험해짐 
#             정보를 감출수록 -> 불편해짐

In [ ]:
# mmr : retriever : Embedding, Bm25, Hybrid, Maximum Marginal Relevance

In [ ]:
# 쿼리 : 하네스 엔지니어링
# -> 쿼리랑 가장 유사한 문서들을 찾아옵니다.

In [ ]:
# 1. 하네스 엔지니어링은 ~~~입니다.
# 2. ~~~
# 3. ~~~

# Relevance <-----> Diversity (다양성)

In [ ]:
# P = 0.1, 0.2, ..
# mmr = P * Relevance(s, query) - (1-P)* max(Similarity(s, selected))

In [ ]:
# retrieved = [1, 2, 3, 4, 5]
# selected = [1]
# K 

In [ ]:
[(Document(id='f941a5a5-8291-41c4-97c4-8719bedacde4', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  1.0),
 (Document(id='0820d82b-7f85-4f20-9278-df128485b595', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  0.9),
 (Document(id='9711ac83-c999-43eb-8556-3dd19c3ec446', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  0.8),
 (Document(id='08c38071-9fc0-463a-a01b-5651b4231e2b', metadata={'id': 'd6'}, page_content='파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.'),
  0.7),
 (Document(id='a81e6fa7-5295-4594-a085-23867b311ae3', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  0.6)]


In [55]:
document = "BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다. 트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다. GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다."
document.split('')

['BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다',
 ' 트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다',
 ' GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다',
 '']

In [56]:
import re
def extractive_compress(query, document, max_sentences=3):
#     sentences = document.split('.')
    sentences = re.split(r'[.!?]\s*', document)
    
    if not sentences:
        return document
    
    query_terms = set(query.lower().split())
    
    scored = []
    for sent in sentences:
        sent_terms = set(sent.lower().split())
        overlap = len(query_terms & sent_terms)
        scored.append((sent, overlap))
        
    scored.sort(key = lambda x: x[1], reverse=True)
    selected = [s for s, _ in scored[:max_sentences]]
    
    return ". ".join(selected) + "."

In [58]:
long_doc = "트랜스포머는 2017년 구글이 발표한 아키텍처입니다. Self-Attention 메커니즘이 핵심입니다. 이전의 RNN, LSTM과 달리 병렬 처리가 가능합니다. BERT와 GPT 모두 트랜스포머를 기반으로 합니다. 자연어 처리뿐 아니라 컴퓨터 비전에서도 활용됩니다"
query = "트랜스포머와 BERT의 관계"
compressed = extractive_compress(query, long_doc, max_sentences=3)
compressed

'트랜스포머는 2017년 구글이 발표한 아키텍처입니다. Self-Attention 메커니즘이 핵심입니다. 이전의 RNN, LSTM과 달리 병렬 처리가 가능합니다.'

In [59]:
compress_chain = ChatPromptTemplate.from_messages([
    ("system", "당신은 문서 요약 전문가입니다"),
    ("human", """다음 문서에서 쿼리에 답하는데 필요한 핵심 정보만 추출하세요.
    불필요한 내용은 제거하고, {max_tokens}자 이내로 압축하세요.
    
    쿼리 : {query}
    문서 : {document}
    
    압축결과:""")
]) | llm | StrOutputParser()

In [60]:
def llm_compress(query, document, max_tokens=100):
    return compress_chain.invoke({
        'query' : query,
        'document' : document,
        'max_tokens' : max_tokens
    })

In [61]:
compressed_llm = llm_compress(query, long_doc)

In [62]:
compressed_llm

'트랜스포머는 2017년 발표된 아키텍처로, BERT와 GPT의 기초가 됩니다.'

In [63]:
compressor = LLMChainExtractor.from_llm(llm)
compressor_retriever = ContextualCompressionRetriever(base_compressor = compressor, 
                                                      base_retriever=vectorstore.as_retriever(search_kwargs={'k':3}))

In [64]:
compressed_docs = compressor_retriever.invoke(query)

In [65]:
compressed_docs

[Document(metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
 Document(metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.')]

In [71]:
# 압축 품질 평가 : 압축률(200자 -> 100자), 키워드 보존율, 의미 보존도 (emb)
def evaluate_compression(original, compressed, query):
    # 1) 압축율
    ratio = len(compressed)  / len(original)
    
    # 2) 키워드 보존율
    query_terms = set(query.lower().split())
    orig_terms = set(original.lower().split())
    comp_terms = set(compressed.lower().split())
    
    orig_query_terms = query_terms & orig_terms
    comp_query_terms = query_terms & comp_terms
    keyword_coverage = len(comp_query_terms) / len(orig_query_terms) if orig_query_terms else 1.0
    
    # 3) 의미 보존도
    original_emb = np.array(embeddings_model.embed_query(original))  # get_embedding(original)
    compressed_emb = get_embedding(compressed)
    
    semantic_similarity = np.dot(original_emb, compressed_emb) / (np.linalg.norm(original_emb) * np.linalg.norm(compressed_emb))
    
    return {
        'compression_ratio' : ratio,
        'keyword_coverage' : keyword_coverage,
        'semantic_similarity' : semantic_similarity
    }

In [72]:
evaluate_compression(long_doc, compressed_llm, query)

{'compression_ratio': 0.2925170068027211,
 'keyword_coverage': 1.0,
 'semantic_similarity': np.float64(0.7428471714685776)}

In [73]:
# P = 0.1, 0.2, ..
# mmr = P * Relevance(s, query) - (1-P)* max(Similarity(s, selected))

In [74]:
def cosine_sim(original_emb, compressed_emb):
    return np.dot(original_emb, compressed_emb) / (np.linalg.norm(original_emb) * np.linalg.norm(compressed_emb))

In [77]:
def mmr_compress(query, document, max_sentences=3, param=0.5):
    sentences = re.split(r'[.!?]\s*', document)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
    
    if len(sentences) <= max_sentences:
        return ". ".join(sentences) + "."
    
    query_emb = get_embedding(query)
    sent_embs = [get_embedding(s) for s in sentences]
    
    selected = []
    remaining = list(range(len(sentences)))
    
    for _ in range(max_sentences):
        best_score = -float('inf')
        best_idx = -1
        
        for idx in remaining:
            relevance = cosine_sim(query_emb, sent_embs[idx])
            
            if selected:
                max_sim = max(cosine_sim(sent_embs[idx], sent_embs[s]) for s in selected)
            else:
                max_sim = 0.0
                
            mmr = param * relevance - (1 - param) * max_sim
            
            if mmr > best_score:
                best_score = mmr
                best_idx = idx
        
        selected.append(best_idx)
        remaining.remove(best_idx)
        
        
    return ". ".join(sentences[i] for i in sorted(selected)) + "."

In [78]:
mmr_compress(query, long_doc, max_sentences=2, param=0.7)

'트랜스포머는 2017년 구글이 발표한 아키텍처입니다. BERT와 GPT 모두 트랜스포머를 기반으로 합니다.'

In [79]:
mmr_compress(query, long_doc, max_sentences=2, param=0.3)

'트랜스포머는 2017년 구글이 발표한 아키텍처입니다. 이전의 RNN, LSTM과 달리 병렬 처리가 가능합니다.'

In [ ]:
리랭킹 효과
Precsion - Average Precision - mAP (mean-Average-Precision)

In [ ]:
BERT         [관련O, 관련X, 관련O, 관련X, 관련O]
Precision     1/1     0     2/3     0     3/5
AP            (1/1      +   2/3     +     3/5) / 3 = 0.7556

In [ ]:
BERT2         [관련O, 관련X, 관련O, 관련X, 관련O]
Precision     1/1      0    2/3     0     3/5
AP            (1/1      +   2/3     +     3/5) / 3 = 0.7777

In [80]:
(1/1      +   2/3     +     3/5) / 3 

0.7555555555555555

In [ ]:
ILS (Intra-List Similarity)        : 상위문서들 간의 유사도 (높을수록 다양성이 낮다)
alpha-NDCG                         : alpha (diversity weighted param)

In [81]:
def average_precision(retrieved_ids, relevant_ids):
    relevant_set = set(relevant_ids)
    hits = 0
    sum_precision = 0.0
    
    for i, doc_id in enumerate(retrieved_ids):
        if doc_id in relevant_set:
            hits += 1
            precision_at_i = hits / (i+1)
            sum_precision += precision_at_i
            
    return sum_precision/len(relevant_set) if relevant_set else 0.0



In [82]:
def mean_average_precision(query_results):
    # query_results : {query1 : (retrieved_ids, relvant_ids), query2 : (retrieved_ids, relvant_ids)}
    
    aps = []
    for query, (retrieved, relevant) in query_results.items():
        ap = average_precision(retrieved, relevant)
        aps.append(ap)
        
    # aps = [0.7556, 0.7777, 0.3333, ,,,,]
    map_score = np.mean(aps)
    return map_score

In [83]:
results

[(Document(id='0820d82b-7f85-4f20-9278-df128485b595', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  np.float32(0.52375495)),
 (Document(id='f941a5a5-8291-41c4-97c4-8719bedacde4', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  np.float32(0.46395132)),
 (Document(id='9711ac83-c999-43eb-8556-3dd19c3ec446', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  np.float32(0.41890207)),
 (Document(id='08c38071-9fc0-463a-a01b-5651b4231e2b', metadata={'id': 'd6'}, page_content='파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.'),
  np.float32(0.41281974)),
 (Document(id='a81e6fa7-5295-4594-a085-23867b311ae3', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  np.float32(0.39723063))]

In [85]:
relevant = {'d1', 'd2', 'd3'}
orig_order = [doc.metadata['id'] for doc, _ in results]
orig_order

['d1', 'd2', 'd3', 'd6', 'd7']

In [86]:
ap = average_precision(orig_order, relevant)

In [87]:
ap

1.0

In [88]:
def intra_list_similarity(doc_ids, doc_embeddings, top_k=5):
    ids = doc_ids[:top_k]
    embs = [doc_embeddings[did] for did in ids if did in doc_embeddings]
    
    if len(embs) < 2:
        return 0.0
    
    sims = []
    for i in range(len(embs)):
        for j in range(i+1, len(embs)):
            sims.append(cosine_sim(embs[i], embs[j]))
            
    return np.mean(sims)

In [90]:
ils_score = intra_list_similarity(orig_order, doc_embeddings)
ils_score

0.0

In [ ]:
[a, b, c, d, e]
(a, b), (a, c), (a, d) (a, e)
        (b, c), (b, d) , 
                (c, d)